# Proyecto RaSA — Entendimiento de datos 2
**Entregable 1 — Calidad de datos**

Se revisas las 4 dimensiones de calidad vistas en el curso en las fuentes compartidas por RaSA en la base `RaSaTransaccional` y saque conclusiones.
- **1. Completitud**
- **2. Unicidad**
- **3. Validez**
- **4. Consistencia**

Las fuentes evaluadas son:

- **F1. FuenteAreasDeServicio_Copia_E** — áreas de servicio por condado.
- **F2. FuenteTiposBeneficio_Copia_E** — tipos de beneficio.
- **F3. FuentePlanesBeneficio_Copia_E** — beneficios ofrecidos por cada plan (tabla central).
- **F4. FuenteCondicionesDePago_Copia_E** — condiciones de pago (referencia).

## 0. Configuración y utilidades

In [2]:
# Credenciales y conexión (use su usuario/clave del curso y su servidor de los tutoriales)
db_user = 'DB_202613_cj_hernandezg12'
db_psswd = '201224455'

# Base de datos de las fuentes
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional'  # AJUSTE host/puerto al suyo

# Driver JDBC de MySQL
path_jar_driver = r'C:\Users\cjhg0\OneDrive\Documentos\CJHG\MAIT\3.2. Modelado de Datos y ETL\mysql-connector-java-8.0.28.jar'

In [8]:
import os
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = (
    SparkSession.builder
    .appName("CalidadDDatos_RaSA")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.jars", path_jar_driver)
    .config("spark.driver.extraClassPath", path_jar_driver)
    .config("spark.executor.extraClassPath", path_jar_driver)
    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [9]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def cargar_tabla(nombre):
    sql = f'(SELECT * FROM RaSaTransaccional.{nombre}) AS t'
    return obtener_dataframe_de_bd(source_db_connection_string, sql, db_user, db_psswd)

In [5]:
# Verifique los nombres reales de las fuentes
tablas_bd = obtener_dataframe_de_bd(source_db_connection_string,
    "(SELECT table_name FROM information_schema.tables WHERE table_schema='RaSaTransaccional' AND table_name LIKE 'Fuente%') AS t",
    db_user, db_psswd)
tablas_bd.show(truncate=False)

TABLAS = {
    'areas':        'FuenteAreasDeServicio_Copia_E',
    'tipos':        'FuenteTiposBeneficio_Copia_E',
    'planes':       'FuentePlanesBeneficio_Copia_E',   # ajuste si en la BD es 'FuentesPlanes...'
    'condiciones':  'FuenteCondicionesDePago_Copia_E',
}

+-------------------------------+
|TABLE_NAME                     |
+-------------------------------+
|FuenteAreasDeServicio_Copia_E  |
|FuenteCondicionesDePago_Copia_E|
|FuentePlanesBeneficio_Copia_E  |
|FuenteTiposBeneficio_Copia_E   |
+-------------------------------+



In [13]:
areas = cargar_tabla(TABLAS['areas'])
areas.show(5)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|    464269.0|805.0|   577.0| 2017|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|    518117.0|185.0|  2801.0| 2018|
|          10132017|BlueOptions16842F...|        12031|     Duval County|   Florida|    999935.0|774.0|  1292.0| 2017|
|         101982018|New Jersey - Medi...|        34003|    Bergen County|New Jersey|    953819.0|234.0|  4076.0| 2018|
|         102012017|New Jersey - Medi...|        34021|    Mercer County|New Jersey|    385898.0|226.0|  1708.0| 2017|
+------------------+--------------------+-------

In [14]:
tipos = cargar_tabla(TABLAS['tipos'])
tipos.show(5)

+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|     UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|              565|Nutritional Couns...|                    |   No|                  Yes|                     No|                                      No|                                    Yes| 2017|
|              795|Rehabilitative Sp...|Days per Benefit ...|  Yes|                  Yes|                    Yes|                                      No|                                    Yes| 2

In [15]:
planes = cargar_tabla(TABLAS['planes'])
planes.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|              640|          10382017|                       34|                         27|                3|16842FL0070128-03|2017-12-31|        16842|          0|           50|            35|
|              225|          31512017|                      238|                         45|                2|29418TX0140002-04|2017-12-31|        29418|          0|            0|          NULL|
|              190|      

In [16]:
condiciones = cargar_tabla(TABLAS['condiciones'])
condiciones.show(5)

+---------------------+--------------------+--------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|
+---------------------+--------------------+--------+
|                  187|Copay with deduct...|  Copago|
|                  204|       Copay per Day|  Copago|
|                   45|         Coinsurance|Coseguro|
|                   85|Copay per Day bef...|  Copago|
|                   18|No Charge after d...|Coseguro|
+---------------------+--------------------+--------+
only showing top 5 rows


## 1. Completitud

La completitud evalúa si los campos requeridos para los análisis de RaSA contienen información suficiente o si presentan valores nulos o vacíos que puedan impedir la integración de fuentes o el cálculo de indicadores.

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

fuentes = {
    "AreasDeServicio": areas,
    "TiposBeneficio": tipos,
    "PlanesBeneficio": planes,
    "CondicionesDePago": condiciones
}

def evaluar_completitud(df, nombre_fuente):
    total_registros = df.count()
    resultados = []

    for columna in df.columns:
        valores_faltantes = df.filter(
            F.col(columna).isNull() |
            (F.trim(F.col(columna).cast("string")) == "")
        ).count()

        porcentaje_faltantes = (
            round((valores_faltantes / total_registros) * 100, 2)
            if total_registros > 0 else 0
        )

        resultados.append((
            nombre_fuente,
            columna,
            total_registros,
            valores_faltantes,
            porcentaje_faltantes
        ))

    return spark.createDataFrame(
        resultados,
        ["Fuente", "Campo", "Total_Registros", "Valores_Faltantes", "Porcentaje_Faltantes"]
    )

In [ ]:
resultados_completitud = []

for nombre_fuente, df in fuentes.items():
    resultados_completitud.append(
        evaluar_completitud(df, nombre_fuente)
    )

completitud = reduce(
    lambda df1, df2: df1.unionByName(df2),
    resultados_completitud
)


In [29]:
completitud_problemas = (
    completitud
    .filter(F.col("Valores_Faltantes") > 0)
    .orderBy(F.desc("Porcentaje_Faltantes"))
)

completitud_problemas.show(100, truncate=False)

+---------------+------------------+---------------+-----------------+--------------------+
|Fuente         |Campo             |Total_Registros|Valores_Faltantes|Porcentaje_Faltantes|
+---------------+------------------+---------------+-----------------+--------------------+
|PlanesBeneficio|cantidadLimite    |36036          |30571            |84.83               |
|TiposBeneficio |UnidadDelLimite   |849            |559              |65.84               |
|PlanesBeneficio|IdTipoBeneficio_T |36036          |2086             |5.79                |
|PlanesBeneficio|IdPlan_T          |36036          |2053             |5.7                 |
|PlanesBeneficio|IdAreaDeServicio_T|36036          |2041             |5.66                |
|AreasDeServicio|IdGeografia_T     |188815         |6378             |3.38                |
|AreasDeServicio|IdAreaDeServicio_T|188815         |6288             |3.33                |
|AreasDeServicio|Densidad          |188815         |2407             |1.27      

In [30]:
resumen_completitud = (
    completitud
    .groupBy("Fuente")
    .agg(
        F.count("Campo").alias("Total_Campos"),
        F.sum(F.when(F.col("Valores_Faltantes") > 0, 1).otherwise(0)).alias("Campos_Con_Faltantes"),
        F.sum("Valores_Faltantes").alias("Total_Valores_Faltantes"),
        F.max("Porcentaje_Faltantes").alias("Mayor_Porcentaje_Faltantes")
    )
    .orderBy("Fuente")
)

resumen_completitud.show(truncate=False)

+-----------------+------------+--------------------+-----------------------+--------------------------+
|Fuente           |Total_Campos|Campos_Con_Faltantes|Total_Valores_Faltantes|Mayor_Porcentaje_Faltantes|
+-----------------+------------+--------------------+-----------------------+--------------------------+
|AreasDeServicio  |9           |4                   |17480                  |3.38                      |
|CondicionesDePago|3           |0                   |0                      |0.0                       |
|PlanesBeneficio  |11          |4                   |36751                  |84.83                     |
|TiposBeneficio   |9           |1                   |559                    |65.84                     |
+-----------------+------------+--------------------+-----------------------+--------------------------+



El análisis de completitud confirma los hallazgos identificados durante el perfilamiento inicial y permite evaluar su impacto sobre la calidad de los datos. Aunque algunas fuentes presentan un buen nivel de completitud, existen campos faltantes que afectan directamente la integración entre tablas y la construcción de análisis confiables.

La fuente `CondicionesDePago` no presenta valores faltantes en los campos evaluados, por lo que no se identifican problemas de completitud en esta fuente. En contraste, `PlanesBeneficio` concentra el mayor volumen de faltantes, principalmente en `cantidadLimite`, con 30.571 registros sin información, equivalentes al 84,83%. Este hallazgo debe analizarse junto con la regla de negocio de beneficios con límite cuantitativo, ya que no todos los beneficios necesariamente requieren una cantidad límite.

En `TiposBeneficio`, el campo `UnidadDelLimite` presenta 559 valores faltantes, equivalentes al 65,84%. Este resultado también debe interpretarse en relación con el atributo `TieneLimiteCuantitativo`, dado que la unidad del límite solo sería obligatoria para beneficios que manejan límites cuantitativos.

Por su parte, `AreasDeServicio` presenta faltantes en `IdGeografia_T`, `IdAreaDeServicio_T`, `Area` y `Densidad`. Aunque los porcentajes son bajos, entre 1,27% y 3,38%, el faltante en `IdAreaDeServicio_T` es especialmente crítico porque este campo permite relacionar las áreas de servicio con la fuente central de planes-beneficio.

En conclusión, la completitud de los datos es parcialmente adecuada para continuar con el proyecto. Sin embargo, antes de construir el modelo dimensional deben definirse reglas para tratar los campos faltantes, especialmente aquellos asociados con llaves de integración, variables geográficas y atributos requeridos para validar límites cuantitativos.

## 2. Unicidad

La unicidad evalúa si los registros o identificadores permiten reconocer de manera única cada entidad dentro de una fuente. En esta sección se revisa la cantidad de registros totales, registros completamente únicos, registros duplicados y valores distintos de las llaves principales identificadas en cada fuente.

Este análisis permite identificar posibles duplicidades, presencia de información histórica o problemas en la definición de llaves operacionales.

In [31]:
def evaluar_unicidad(df, nombre_fuente, llave):
    total_registros = df.count()
    registros_unicos = df.distinct().count()
    duplicados_completos = total_registros - registros_unicos

    valores_distintos_llave = df.select(llave).distinct().count()

    llaves_repetidas = (
        df
        .groupBy(llave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    porcentaje_duplicados_completos = (
        round((duplicados_completos / total_registros) * 100, 2)
        if total_registros > 0 else 0
    )

    return spark.createDataFrame(
        [(
            nombre_fuente,
            llave,
            total_registros,
            registros_unicos,
            duplicados_completos,
            porcentaje_duplicados_completos,
            valores_distintos_llave,
            llaves_repetidas
        )],
        [
            "Fuente",
            "Llave_Evaluada",
            "Total_Registros",
            "Registros_Unicos",
            "Duplicados_Completos",
            "Porcentaje_Duplicados_Completos",
            "Valores_Distintos_Llave",
            "Llaves_Repetidas"
        ]
    )

In [32]:
unicidad_areas = evaluar_unicidad(
    areas,
    "AreasDeServicio",
    "IdAreaDeServicio_T"
)

unicidad_tipos = evaluar_unicidad(
    tipos,
    "TiposBeneficio",
    "IdTipoBeneficio_T"
)

unicidad_planes = evaluar_unicidad(
    planes,
    "PlanesBeneficio",
    "IdPlan_T"
)

unicidad_condiciones = evaluar_unicidad(
    condiciones,
    "CondicionesDePago",
    "IdCondicionesDePago_T"
)

In [33]:
unicidad = (
    unicidad_areas
    .unionByName(unicidad_tipos)
    .unionByName(unicidad_planes)
    .unionByName(unicidad_condiciones)
)

unicidad.show(truncate=False)

+-----------------+---------------------+---------------+----------------+--------------------+-------------------------------+-----------------------+----------------+
|Fuente           |Llave_Evaluada       |Total_Registros|Registros_Unicos|Duplicados_Completos|Porcentaje_Duplicados_Completos|Valores_Distintos_Llave|Llaves_Repetidas|
+-----------------+---------------------+---------------+----------------+--------------------+-------------------------------+-----------------------+----------------+
|AreasDeServicio  |IdAreaDeServicio_T   |188815         |145242          |43573               |23.08                          |5410                   |5384            |
|TiposBeneficio   |IdTipoBeneficio_T    |849            |578             |271                 |31.92                          |178                    |132             |
|PlanesBeneficio  |IdPlan_T             |36036          |27543           |8493                |23.57                          |393                    |392 

In [34]:
areas_repetidas = (
    areas
    .groupBy("IdAreaDeServicio_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

areas_repetidas.show(20, truncate=False)

+------------------+-----+
|IdAreaDeServicio_T|count|
+------------------+-----+
|NULL              |6288 |
|92882017          |237  |
|92932017          |230  |
|92892017          |213  |
|92872017          |204  |
|92942017          |189  |
|92912017          |186  |
|76462018          |185  |
|76422017          |184  |
|33482017          |183  |
|92922017          |177  |
|24232018          |170  |
|51242017          |169  |
|33432017          |168  |
|33292017          |164  |
|33372017          |164  |
|51452017          |163  |
|51512017          |161  |
|51212017          |160  |
|33312017          |160  |
+------------------+-----+
only showing top 20 rows


In [35]:
tipos_repetidos = (
    tipos
    .groupBy("IdTipoBeneficio_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

tipos_repetidos.show(20, truncate=False)

+-----------------+-----+
|IdTipoBeneficio_T|count|
+-----------------+-----+
|190              |29   |
|865              |26   |
|595              |21   |
|825              |20   |
|830              |19   |
|335              |19   |
|765              |19   |
|275              |18   |
|565              |17   |
|135              |17   |
|455              |16   |
|835              |16   |
|1000             |15   |
|820              |15   |
|10               |15   |
|640              |15   |
|795              |14   |
|330              |14   |
|1030             |14   |
|85               |13   |
+-----------------+-----+
only showing top 20 rows


In [36]:
condiciones_repetidas = (
    condiciones
    .groupBy("IdCondicionesDePago_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

condiciones_repetidas.show(20, truncate=False)

+---------------------+-----+
|IdCondicionesDePago_T|count|
+---------------------+-----+
|204                  |3    |
|27                   |2    |
|207                  |2    |
|51                   |2    |
|45                   |2    |
|221                  |2    |
|119                  |2    |
+---------------------+-----+



In [37]:
planes_repetidos = (
    planes
    .groupBy("IdPlan_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

planes_repetidos.show(20, truncate=False)

+-----------------+-----+
|IdPlan_T         |count|
+-----------------+-----+
|                 |2053 |
|40047MI0070001-00|1240 |
|91762NJ0070093-00|613  |
|11512NC0100028-03|546  |
|38166WI0180012-06|543  |
|66252TX0030004-02|517  |
|96751NH0150027-06|508  |
|38345WI0080018-03|505  |
|89942GA0060004-01|495  |
|91661NJ2340003-05|494  |
|37833WI0540010-01|486  |
|29418TX0140002-04|466  |
|11269WY0080014-00|302  |
|49046GA0420043-01|292  |
|36194FL0130003-01|289  |
|96667ME0260012-03|288  |
|99248TN0060001-01|285  |
|36194FL0140005-04|282  |
|90714MS0010005-04|281  |
|19636LA0590003-05|279  |
+-----------------+-----+
only showing top 20 rows


In [38]:
llave_compuesta_planes = [
    "IdPlan_T",
    "IdTipoBeneficio_T",
    "IdAreaDeServicio_T",
    "IdProveedor_T",
    "Fecha"
]

total_planes = planes.count()

planes_unicos_compuesta = (
    planes
    .select(llave_compuesta_planes)
    .distinct()
    .count()
)

duplicados_llave_compuesta = total_planes - planes_unicos_compuesta

print("Total registros PlanesBeneficio:", total_planes)
print("Registros únicos según llave compuesta:", planes_unicos_compuesta)
print("Duplicados según llave compuesta:", duplicados_llave_compuesta)

Total registros PlanesBeneficio: 36036
Registros únicos según llave compuesta: 18682
Duplicados según llave compuesta: 17354


In [39]:
planes_duplicados_compuesta = (
    planes
    .groupBy(llave_compuesta_planes)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

planes_duplicados_compuesta.show(20, truncate=False)

+-----------------+-----------------+------------------+-------------+----------+-----+
|IdPlan_T         |IdTipoBeneficio_T|IdAreaDeServicio_T|IdProveedor_T|Fecha     |count|
+-----------------+-----------------+------------------+-------------+----------+-----+
|38345WI0080018-03|NULL             |50152017          |38345        |2017-12-31|28   |
|87571OK0350027-05|NULL             |95082017          |87571        |2017-12-31|25   |
|67243LA0100009-01|NULL             |82642017          |67243        |2017-12-31|22   |
|33602TX0460570-01|NULL             |37932018          |33602        |2018-12-31|20   |
|37833WI0380063-01|NULL             |45922017          |37833        |2017-12-31|20   |
|36194FL0130003-01|NULL             |44112018          |36194        |2018-12-31|16   |
|37833WI0540010-01|NULL             |46912018          |37833        |2018-12-31|16   |
|37833WI0540010-01|NULL             |46922017          |37833        |2017-12-31|15   |
|67243LA0090013-05|NULL         

### Análisis de unicidad

El análisis de unicidad evidencia que ninguna de las llaves evaluadas identifica de forma única cada fila de su respectiva fuente. Todas las fuentes presentan registros duplicados completos y repetición de identificadores, lo cual confirma que existen problemas de duplicidad o que algunas fuentes contienen información histórica o de detalle que no puede ser identificada únicamente con una llave simple.

En `AreasDeServicio`, la fuente contiene 188.815 registros, pero solo 5.410 valores distintos de `IdAreaDeServicio_T`. Este resultado confirma que el identificador del área de servicio se repite múltiples veces dentro de la tabla. Además, se identifican 43.573 registros duplicados completos, equivalentes al 23,08% del total. La diferencia frente a los 5.409 valores esperados por el negocio puede explicarse por la existencia de valores nulos en la llave, identificados previamente en el análisis de completitud.

En `TiposBeneficio`, se observan 849 registros, 578 registros únicos y 271 duplicados completos, equivalentes al 31,92%. Adicionalmente, la llave `IdTipoBeneficio_T` presenta 178 valores distintos y 132 identificadores repetidos. Esto indica que un mismo tipo de beneficio aparece más de una vez, posiblemente por variaciones temporales, diferencias en atributos descriptivos o duplicidad operacional.

En `CondicionesDePago`, aunque la fuente tiene un volumen bajo de registros, también se identifican problemas de unicidad. De 31 registros, solo 24 son únicos y existen 7 duplicados completos, equivalentes al 22,58%. La llave `IdCondicionesDePago_T` presenta 23 valores distintos y 7 identificadores repetidos, por lo que se requiere depuración antes de usar esta fuente como dimensión de referencia.

En `PlanesBeneficio`, la repetición de `IdPlan_T` debe interpretarse de manera diferente. La fuente contiene 36.036 registros y solo 393 planes distintos, pero esto no necesariamente representa un error, ya que un mismo plan puede estar asociado a múltiples beneficios, áreas de servicio, proveedores, niveles de servicio y fechas. Sin embargo, se identifican 8.493 duplicados completos, equivalentes al 23,57%, lo cual sí sugiere que existen registros exactamente repetidos que deberían ser revisados o eliminados antes de construir el modelo dimensional.

En conclusión, la unicidad de los datos no es completamente adecuada para el modelado analítico sin aplicar reglas de depuración. Se recomienda eliminar duplicados completos, revisar la definición de llaves para cada fuente y, especialmente en `PlanesBeneficio`, utilizar una llave compuesta que represente mejor el nivel de detalle de la tabla.

## 3. Validez

## 4. Consistencia

## 5. Conclusiones